# News Quality Analysis — Duplicate & Bad Content Detection

**Goal:** Identify and flag articles that should be removed from the dataset:
1. **Duplicates** — Articles with identical or near-identical text within the same newspaper
2. **Blocked/Captcha pages** — Pages where the scraper captured a paywall, cookie consent, or captcha page instead of actual content
3. **Boilerplate-only** — Articles that are mostly navigation text, footer content, or generic templates

**Pipeline:**
1. Load all articles from `../data-scraping/data/`
2. Detect exact duplicates (same text hash within same domain)
3. Detect near-duplicates using MinHash/LSH (within same domain)
4. Pattern-based detection of bad content (blocked pages, captchas, cookie banners)
5. Short/empty content detection
6. Output a JSONL file with article IDs to remove and the reason

**Output:** `output/articles_to_remove.jsonl` — one line per article: `{id, domain, reason}`

## Step 0: Install Dependencies

In [ ]:
# %pip install polars datasketch tqdm

## Step 1: Load Articles

In [ ]:
import polars as pl
from pathlib import Path
import time

DATA_DIR = Path("../data-scraping/data")
JORNAIS_CSV = Path("../data-scraping/jornais.csv")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

# Load the list of known newspapers and extract their domains
jornais = pl.read_csv(JORNAIS_CSV)
known_domains = set(
    jornais["url"]
    .drop_nulls()
    .str.replace(r"^https?://", "")
    .str.replace(r"^www\.", "")
    .str.strip_chars("/")
    .to_list()
)
print(f"Known domains from jornais.csv: {len(known_domains)}")

# Scan all articles.jsonl files
t0 = time.time()
jsonl_files = sorted(DATA_DIR.glob("*/articles.jsonl"))
print(f"Found {len(jsonl_files)} articles.jsonl files")

ALL_UTF8 = {col: pl.Utf8 for col in [
    "id", "url", "title", "subtitle", "text", "date", "author",
    "agency", "section", "source", "timestamp", "archive_url",
    "domain", "text_hash", "fetched_at", "extractor",
]}

# Load only required columns, filter to known domains immediately per file
# This avoids holding all text in a single giant DataFrame
dfs = []
skipped = 0
for f in jsonl_files:
    try:
        df = pl.read_ndjson(f, schema_overrides=ALL_UTF8).select(
            ["id", "title", "text", "domain"]
        )
        # Filter to known domains immediately to reduce memory
        df = df.filter(
            pl.col("domain")
            .str.replace(r"^www\.", "")
            .str.strip_chars("/")
            .is_in(known_domains)
        )
        if df.height > 0:
            dfs.append(df)
    except Exception as e:
        skipped += 1

articles = pl.concat(dfs)
del dfs
elapsed = time.time() - t0
print(f"Loaded {len(articles):,} articles in {elapsed:.1f}s (skipped {skipped} files)")

In [ ]:
# Combine title + text, then immediately drop the source columns to save RAM
articles = articles.with_columns(
    (pl.col("title").fill_null("") + " " + pl.col("text").fill_null(""))
    .str.strip_chars()
    .alias("full_text")
).drop(["title", "text"])

# Basic stats
print(f"Articles from known jornais: {len(articles):,}")
print(f"Articles with empty/very short text (<50 chars): {articles.filter(pl.col('full_text').str.len_chars() < 50).height:,}")
print(f"Unique domains: {articles['domain'].n_unique()}")
print(f"Approx DataFrame size: {articles.estimated_size('mb'):.0f} MB")

In [ ]:
import gc
del jornais, known_domains, jsonl_files
gc.collect()
print(f"Ready. Articles shape: {articles.shape}")

## Step 2: Exact Duplicates (within same newspaper)

Articles with identical full_text within the same domain are exact duplicates.
We keep the first occurrence and flag the rest.

In [ ]:
# Compute text hash using Polars native hashing (no Python UDF needed)
articles = articles.with_columns(
    pl.col("full_text").hash().alias("text_hash")
)

# Find duplicates: same domain + same text hash
# Use Polars group_by to find groups with >1 row, then keep only the first
dup_groups = (
    articles
    .with_row_index("row_idx")
    .group_by(["domain", "text_hash"])
    .agg([
        pl.col("id"),
        pl.col("row_idx").first().alias("first_idx"),
        pl.len().alias("group_size"),
    ])
    .filter(pl.col("group_size") > 1)
)

# Explode to get all IDs in duplicate groups, then remove the first occurrence
dup_all = dup_groups.explode("id").select(["id", "first_idx", "group_size"])

# The first ID in each group is kept; we need to identify the "non-first" ones
# Re-do: get (domain, text_hash, row_idx, id) for rows in dup groups only
dup_hashes = dup_groups.select(["domain", "text_hash"]).unique()
dup_rows = (
    articles.with_row_index("row_idx")
    .join(dup_hashes, on=["domain", "text_hash"], how="inner")
)

# Keep the first row_idx per group, flag the rest
first_per_group = dup_rows.group_by(["domain", "text_hash"]).agg(
    pl.col("row_idx").min().alias("keep_idx")
)
dup_rows_flagged = dup_rows.join(first_per_group, on=["domain", "text_hash"])
duplicate_ids = set(
    dup_rows_flagged.filter(pl.col("row_idx") != pl.col("keep_idx"))["id"].to_list()
)

n_groups = dup_groups.height
print(f"Exact duplicate groups: {n_groups:,}")
print(f"Articles to remove (exact duplicates): {len(duplicate_ids):,}")

# Clean up intermediate frames
del dup_groups, dup_all, dup_hashes, dup_rows, first_per_group, dup_rows_flagged
gc.collect()

## Step 3: Near-Duplicates (MinHash LSH within same newspaper)

Detect articles with very similar text (e.g. same article with minor edits, different timestamps).
Uses MinHash with Jaccard similarity threshold of 0.8.

In [ ]:
# from datasketch import MinHash, MinHashLSH
# from tqdm.auto import tqdm
# import re
#
# def text_to_shingles(text: str, k: int = 3) -> set:
#     """Convert text to word k-shingles (much smaller than character shingles)."""
#     words = re.sub(r'\s+', ' ', text.lower().strip()).split()
#     if len(words) < k:
#         return set()
#     return {' '.join(words[i:i+k]) for i in range(len(words) - k + 1)}
#
# def make_minhash(shingles: set, num_perm: int = 64) -> MinHash:
#     """Create a MinHash from a set of shingles."""
#     m = MinHash(num_perm=num_perm)
#     for s in shingles:
#         m.update(s.encode('utf-8'))
#     return m
#
# # Parameters — reduced num_perm from 128 to 64 (halves MinHash memory)
# SIMILARITY_THRESHOLD = 0.8
# NUM_PERM = 64
#
# print(f"Near-duplicate detection config:")
# print(f"  Threshold: {SIMILARITY_THRESHOLD}")
# print(f"  Permutations: {NUM_PERM}")
# print(f"  Shingle type: word 3-grams")

# Skipping near-duplicate detection (requires datasketch)
near_duplicate_ids = set()

In [ ]:
# # Near-duplicate detection per domain using LSH
# # Process one domain at a time to limit peak memory
# # Skip articles already flagged as exact duplicates or too short (<100 chars)
#
# # Pre-filter the working set: exclude exact duplicates and very short texts
# work = articles.filter(
#     (~pl.col("id").is_in(duplicate_ids)) &
#     (pl.col("full_text").str.len_chars() > 100)
# )
#
# # Get domain counts for progress bar
# domain_counts = work.group_by("domain").agg(pl.len().alias("n")).filter(pl.col("n") > 1).sort("n", descending=True)
# print(f"Domains with >1 eligible article: {domain_counts.height:,}")
# print(f"Top domains by article count:")
# print(domain_counts.head(10))
#
# near_duplicate_ids = set()
# processed_domains = 0
#
# for dom_row in tqdm(domain_counts.iter_rows(named=True), total=domain_counts.height):
#     domain = dom_row["domain"]
#
#     # Extract just the columns we need for this domain
#     domain_df = work.filter(pl.col("domain") == domain).select(["id", "full_text"])
#
#     if domain_df.height <= 1:
#         continue
#
#     lsh = MinHashLSH(threshold=SIMILARITY_THRESHOLD, num_perm=NUM_PERM)
#
#     ids = domain_df["id"].to_list()
#     texts = domain_df["full_text"].to_list()
#     del domain_df  # free the slice
#
#     for art_id, text in zip(ids, texts):
#         shingles = text_to_shingles(text)
#         if not shingles:
#             continue
#         mh = make_minhash(shingles, NUM_PERM)
#         del shingles  # free immediately
#
#         # Query before insert
#         if lsh.query(mh):
#             near_duplicate_ids.add(art_id)
#         else:
#             try:
#                 lsh.insert(art_id, mh)
#             except ValueError:
#                 pass
#
#     del lsh, ids, texts
#     processed_domains += 1
#
#     # Periodic GC for very large runs
#     if processed_domains % 200 == 0:
#         gc.collect()
#
# del work
# gc.collect()
#
# print(f"\nProcessed {processed_domains} domains")
# print(f"Near-duplicate articles found: {len(near_duplicate_ids):,}")

## Step 4: Bad Content Detection (Pattern-Based)

Detect articles that are actually blocked pages, cookie consent pages, captchas,
paywalls, or other non-news content. Uses keyword pattern matching.

In [ ]:
# Patterns that indicate bad content (blocked pages, cookie banners, etc.)
# Each pattern has a name and a list of indicator phrases
BAD_CONTENT_PATTERNS = {
    "cookie_banner": [
        "este site utiliza cookies",
        "utilizamos cookies",
        "we use cookies",
        "this website uses cookies",
        "aceitar cookies",
        "accept cookies",
        "política de cookies",
        "cookie policy",
        "melhor experiência de navegação",
        "melhor experiência no nosso site",
    ],
    # "paywall": [
    #     "para continuar a ler",
    #     "conteúdo exclusivo para assinantes",
    #     "faça login para continuar",
    #     "subscreva para ler",
    #     "artigo exclusivo",
    #     "assine já",
    #     "assine agora",
    #     "conteúdo reservado a assinantes",
    #     "este conteúdo é exclusivo",
    #     "já é assinante",
    # ],
    "captcha_blocked": [
        "verificar que não é um robô",
        "prove that you are not a robot",
        "please verify you are a human",
        "captcha",
        "access denied",
        "acesso negado",
        "forbidden",
        "403 error",
        "the page you requested",
        "you have been blocked",
    ],
    "generic_boilerplate": [
        "javascript is required",
        "enable javascript",
        "ativar javascript",
        "o seu navegador não suporta",
        "your browser does not support",
        "this page requires",
        "por favor atualize o seu navegador",
        "please update your browser",
    ],
    "login_page": [
        "introduza o seu email",
        "introduza a sua password",
        "iniciar sessão",
        "criar conta",
        "recuperar password",
        "esqueceu-se da password",
        "registe-se gratuitamente",
        "área de cliente",
    ],
}

# Flatten all patterns for efficient matching
all_patterns = []
for category, phrases in BAD_CONTENT_PATTERNS.items():
    for phrase in phrases:
        all_patterns.append((category, phrase.lower()))

print(f"Bad content categories: {len(BAD_CONTENT_PATTERNS)}")
print(f"Total pattern phrases: {len(all_patterns)}")
for cat, phrases in BAD_CONTENT_PATTERNS.items():
    print(f"  {cat}: {len(phrases)} phrases")

In [ ]:
# Detect bad content using vectorized Polars operations (no Python loop over rows)
# Exclude already-flagged articles
candidates = articles.filter(
    (~pl.col("id").is_in(duplicate_ids)) &
    (~pl.col("id").is_in(near_duplicate_ids))
).select(["id", "full_text"])

text_lower = candidates.with_columns(
    pl.col("full_text").str.to_lowercase().alias("text_lc"),
    pl.col("full_text").str.len_chars().alias("text_len"),
)

# For each category, count how many phrases match (vectorized)
bad_content_ids = {}

for category, phrases in BAD_CONTENT_PATTERNS.items():
    # Build a column that counts matches for this category
    match_expr = pl.lit(0)
    for phrase in phrases:
        match_expr = match_expr + pl.col("text_lc").str.contains(phrase, literal=True).cast(pl.Int32)
    
    flagged = text_lower.with_columns(match_expr.alias("n_matches")).filter(
        # Flag if >=2 matches AND (text < 500 chars OR >=3 matches)
        (pl.col("n_matches") >= 2) &
        ((pl.col("text_len") < 500) | (pl.col("n_matches") >= 3))
    )
    
    for art_id in flagged["id"].to_list():
        if art_id not in bad_content_ids:
            bad_content_ids[art_id] = f"bad_content:{category}"

del candidates, text_lower
gc.collect()

print(f"\nBad content articles detected: {len(bad_content_ids):,}")
for cat in BAD_CONTENT_PATTERNS:
    count = sum(1 for reason in bad_content_ids.values() if cat in reason)
    print(f"  {cat}: {count:,}")

## Step 5: Short/Empty Content Detection

Flag articles where the actual content is too short to be a real news article.
Threshold: less than 100 characters of meaningful text.

In [ ]:
# Flag very short articles — fully vectorized
MIN_CONTENT_LENGTH = 100

already_flagged = duplicate_ids | near_duplicate_ids | set(bad_content_ids.keys())

short_content_ids = set(
    articles.filter(
        (~pl.col("id").is_in(already_flagged)) &
        (pl.col("full_text").str.len_chars() < MIN_CONTENT_LENGTH)
    )["id"].to_list()
)

print(f"Short content articles (<{MIN_CONTENT_LENGTH} chars): {len(short_content_ids):,}")
del already_flagged

## Step 6: Summary & Output

Combine all flags and output the final removal list.

In [ ]:
# Build final removal list efficiently using a single join (not per-id filtering)
removal_records = (
    [(aid, "exact_duplicate") for aid in duplicate_ids] +
    [(aid, "near_duplicate") for aid in near_duplicate_ids] +
    [(aid, reason) for aid, reason in bad_content_ids.items()] #+
    # [(aid, "short_content") for aid in short_content_ids]
)

removals_df = pl.DataFrame(removal_records, schema=["id", "reason"], orient="row")

# Join with articles to get domain (single vectorized join instead of per-id lookup)
removals_df = removals_df.join(
    articles.select(["id", "domain"]).unique(subset=["id"]),
    on="id",
    how="left"
)

print(f"Total articles to remove: {len(removals_df):,}")
print(f"\nBreakdown by reason:")
print(removals_df.group_by("reason").agg(pl.len().alias("count")).sort("count", descending=True))
print(f"\nTotal articles in dataset: {len(articles):,}")
print(f"Removal rate: {len(removals_df)/len(articles)*100:.1f}%")

In [ ]:
# Save output
import json

output_path = OUTPUT_DIR / "articles_to_remove.jsonl"
with open(output_path, "w") as f:
    removals_df = removals_df.filter(pl.col("reason") != "bad_content:paywall")
    for row in removals_df.iter_rows(named=True):
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Saved {len(removals_df):,} removal entries to {output_path}")

# Also save a summary CSV per domain
domain_summary = (
    removals_df
    .group_by(["domain", "reason"])
    .agg(pl.len().alias("count"))
    .sort(["domain", "count"], descending=[False, True])
)
domain_summary.write_csv(OUTPUT_DIR / "removal_summary_by_domain.csv")
print(f"Saved domain summary to {OUTPUT_DIR / 'removal_summary_by_domain.csv'}")

## Step 7: Sample Review

Display some examples from each category to verify the detection is working correctly.

In [ ]:
# Show examples from each removal reason (sample 3 per category)
reasons = ["exact_duplicate", "near_duplicate", "short_content"] + [f"bad_content:{cat}" for cat in BAD_CONTENT_PATTERNS]

for reason in reasons:
    sample = removals_df.filter(pl.col("reason") == reason).head(3)
    if sample.height == 0:
        continue
    total = removals_df.filter(pl.col("reason") == reason).height
    print(f"\n{'='*60}")
    print(f"REASON: {reason} ({total:,} total)")
    print(f"{'='*60}")
    
    # Get the text for these sample IDs via join
    sample_with_text = sample.join(
        articles.select(["id", "full_text"]),
        on="id",
        how="left"
    )
    for row in sample_with_text.iter_rows(named=True):
        text = (row["full_text"] or "")[:200]
        print(f"\n  [{row['domain']}] {row['id']}")
        print(f"  Text: {text}...")
        print()